# A domicilio — Análisis y motor de recomendación (DHRD)

Notebook único del proyecto de Datos Masivos. Trabaja sobre el **Delivery Hero
Recommendation Dataset (DHRD)** y cubre, en orden:

1. Carga y validación de los datos
2. Exploración (unidad 1 y 2: cola larga, horarios, canastas)
3. Tasa de repetición: ¿lo de siempre o algo nuevo?
4. Elementos frecuentes: A Priori, FP-Growth y PrefixSpan
5. Análisis de vínculos: PageRank por método de potencias, HITS y caminatas aleatorias
6. Comunidades en el grafo de restaurantes
7. Modelos de recomendación y evaluación (precision@k, recall@k, NDCG@k)
8. Capa "repetir o explorar" y arranque en frío del negocio pequeño

Todos los resultados numéricos se guardan en `export/resultados.json` y las
figuras en `figs/`.

In [1]:
import json
import os
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

# --- Configuración -----------------------------------------------------------
CITY = os.environ.get("DHRD_CITY", "sg")           # sg = Singapur, se = Estocolmo, tw = Taiwán
DATA_DIR = Path(os.environ.get("DHRD_DIR", "../Data/DHRD"))
SMOKE = os.environ.get("DHRD_SMOKE", "0") == "1"   # modo rápido para probar el notebook
TOP_K = 10
SEED = 42
N_EVAL_USERS = 2_000 if SMOKE else 20_000          # usuarios de prueba para evaluar
N_SEQ_USERS = 3_000 if SMOKE else 30_000           # secuencias para PrefixSpan
rng = np.random.default_rng(SEED)

OUT = Path("export"); OUT.mkdir(exist_ok=True)
FIGS = Path("figs"); FIGS.mkdir(exist_ok=True)
R = {}   # aquí se van guardando todos los resultados

CIUDAD = {"sg": "Singapur", "se": "Estocolmo", "tw": "Taiwán"}[CITY]
print(f"Ciudad: {CIUDAD} ({CITY}) · modo rápido: {SMOKE}")

Ciudad: Singapur (sg) · modo rápido: False


## Identidad visual

Las gráficas usan la paleta del sistema de diseño **A domicilio**. La serie
categórica es jitomate → cobalto → naranja → violeta, en ese orden fijo; mostaza
y lima solo se usan como relleno con etiqueta directa, porque sobre papel no
alcanzan 3:1 de contraste.

In [2]:
JITOMATE, COBALTO, NARANJA, VIOLETA = "#dc4033", "#2e5193", "#e06129", "#5547b7"
MOSTAZA, LIMA, HOJA, BETABEL = "#f9d949", "#98c257", "#234525", "#bd2a6a"
PAPEL, KRAFT, CARBON, INK_MUTED, HAIRLINE = "#fffcdf", "#f2ede1", "#1c181a", "#5e5446", "#d9d2b8"
SERIE = [JITOMATE, COBALTO, NARANJA, VIOLETA]

plt.rcParams.update({
    "figure.facecolor": PAPEL, "axes.facecolor": PAPEL, "savefig.facecolor": PAPEL,
    "font.family": "sans-serif",
    "font.sans-serif": ["Montserrat", "DejaVu Sans", "Arial"],
    "axes.titlecolor": CARBON,
    "text.color": CARBON, "axes.labelcolor": CARBON,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": HAIRLINE, "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": HAIRLINE, "grid.linewidth": 0.8,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlepad": 14, "font.size": 10, "legend.frameon": False,
    "figure.dpi": 130, "savefig.bbox": "tight",
})

def guardar(fig, nombre):
    """Guarda la figura en figs/ y la deja visible en el notebook."""
    ruta = FIGS / f"{nombre}.png"
    fig.savefig(ruta, dpi=200)
    print("→", ruta)
    return ruta

def titulo_ax(ax, texto):
    """Título de gráfica: Big Shoulders Display en mayúsculas, como en el sistema."""
    ax.set_title(texto.upper(), fontfamily="Big Shoulders Display", fontsize=16,
                 color=CARBON, loc="left", pad=14)

def miles(x):
    return f"{x:,.0f}"

def pct(x):
    return f"{x:.2f}%"

## 1. Carga y validación

Cada archivo de pedidos trae **una fila por producto**, no por pedido: la llave
del pedido es `order_id`. El corte de entrenamiento y prueba del DHRD es
temporal, así que respetamos el que viene en el dataset (no se mezclan días).

In [3]:
CITY_DIR = DATA_DIR / f"data_{CITY}"
DTYPES = {"customer_id": "category", "geohash": "category", "vendor_id": "category",
          "product_id": "category", "order_id": "int32", "day_of_week": "int8"}

t0 = time.time()
lineas_tr = pd.read_csv(CITY_DIR / f"orders_{CITY}_train.txt", index_col=0, dtype=DTYPES)
lineas_te = pd.read_csv(CITY_DIR / f"orders_{CITY}_test.txt", index_col=0, dtype=DTYPES)
vendors = pd.read_csv(CITY_DIR / f"vendors_{CITY}.txt", index_col=0).drop_duplicates("vendor_id")
products = pd.read_csv(CITY_DIR / f"products_{CITY}.txt", index_col=0)
print(f"Carga en {time.time()-t0:.1f} s")

for df in (lineas_tr, lineas_te):
    df["dia"] = df["order_day"].str.replace(" days", "", regex=False).astype("int16")
    df["hora"] = df["order_time"].str.slice(0, 2).astype("int8")

print("Filas (producto-pedido):", miles(len(lineas_tr)), "train ·", miles(len(lineas_te)), "test")
print("Días:", lineas_tr.dia.min(), "-", lineas_tr.dia.max(), "(train) ·",
      lineas_te.dia.min(), "-", lineas_te.dia.max(), "(test)")

# Validación: un order_id pertenece a un solo cliente
chequeo = lineas_tr.groupby("order_id", observed=True)["customer_id"].nunique()
assert chequeo.max() == 1, "order_id no identifica al pedido"
print("order_id identifica un pedido de un solo cliente ✓")

Carga en 12.4 s


Filas (producto-pedido): 3,431,870 train · 591,135 test
Días: 0 - 75 (train) · 76 - 90 (test)


order_id identifica un pedido de un solo cliente ✓


In [4]:
def a_pedidos(lineas):
    """Colapsa las filas producto-pedido a una fila por pedido."""
    ped = (lineas.groupby("order_id", observed=True)
           .agg(customer_id=("customer_id", "first"),
                vendor_id=("vendor_id", "first"),
                geohash=("geohash", "first"),
                dia=("dia", "first"),
                dow=("day_of_week", "first"),
                hora=("hora", "first"),
                n_productos=("product_id", "size"))
           .reset_index())
    return ped

ped_tr = a_pedidos(lineas_tr)
ped_te = a_pedidos(lineas_te)

# ¿Un pedido puede mezclar restaurantes?
mezcla = lineas_tr.groupby("order_id", observed=True)["vendor_id"].nunique(dropna=True)
R["pedidos_multi_vendor"] = int((mezcla > 1).sum())

R["ciudad"] = CIUDAD
R["pedidos_train"] = int(len(ped_tr))
R["pedidos_test"] = int(len(ped_te))
R["usuarios_train"] = int(ped_tr.customer_id.nunique())
R["usuarios_test"] = int(ped_te.customer_id.nunique())
R["vendors"] = int(vendors.vendor_id.nunique())
R["productos"] = int(products.product_id.nunique())
R["cocinas"] = int(vendors.primary_cuisine.nunique())
R["lineas_totales"] = int(len(lineas_tr) + len(lineas_te))
R["productos_por_pedido"] = float(ped_tr.n_productos.mean())
R["dias_train"] = int(ped_tr.dia.max() - ped_tr.dia.min() + 1)
R["dias_test"] = int(ped_te.dia.max() - ped_te.dia.min() + 1)

pd.Series({k: R[k] for k in ["pedidos_train", "pedidos_test", "usuarios_train",
                             "usuarios_test", "vendors", "productos", "cocinas"]})

pedidos_train     1709414
pedidos_test       290586
usuarios_train     476150
usuarios_test      171515
vendors              7411
productos         1066823
cocinas                78
dtype: int64

## 2. Exploración

Tres preguntas: ¿qué tan concentrada está la demanda?, ¿cuándo se pide?, y
¿qué tan dispersa es la matriz usuario–restaurante?

In [5]:
# --- Cola larga: qué porcentaje de pedidos concentra el top de restaurantes ---
por_vendor = ped_tr.vendor_id.value_counts()
acum = por_vendor.cumsum() / por_vendor.sum()
frac_vendors = np.arange(1, len(por_vendor) + 1) / len(por_vendor)

top10 = float(acum.iloc[max(int(len(por_vendor) * 0.10) - 1, 0)] * 100)
top20 = float(acum.iloc[max(int(len(por_vendor) * 0.20) - 1, 0)] * 100)
R["concentracion_top10_vendors"] = top10
R["concentracion_top20_vendors"] = top20
print(f"El 10% de los restaurantes concentra {pct(top10)} de los pedidos")
print(f"El 20% concentra {pct(top20)}")

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.plot(frac_vendors * 100, acum.values * 100, color=JITOMATE, lw=2)
ax.plot([0, 100], [0, 100], color=HAIRLINE, lw=1.5, ls="--")
ax.annotate(f"10% de los restaurantes\n= {pct(top10)} de los pedidos",
            xy=(10, top10), xytext=(26, top10 - 26), color=CARBON, fontsize=9,
            arrowprops=dict(arrowstyle="-", color=INK_MUTED, lw=1))
ax.set_xlabel("Restaurantes acumulados (%)"); ax.set_ylabel("Pedidos acumulados (%)")
titulo_ax(ax, "La demanda se concentra en pocos restaurantes")
ax.grid(axis="y"); ax.set_axisbelow(True)
guardar(fig, "01_cola_larga"); plt.show()

El 10% de los restaurantes concentra 23.95% de los pedidos
El 20% concentra 46.68%


→ figs/01_cola_larga.png


In [6]:
# --- Ritmo semanal y por hora ---
horas = ped_tr.hora.value_counts().sort_index()
horas_pct = horas / horas.sum() * 100
dias_nombre = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
dow = ped_tr.dow.value_counts().sort_index()
dow_pct = dow / dow.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
axes[0].bar(horas_pct.index, horas_pct.values, color=COBALTO, width=0.75)
titulo_ax(axes[0], "Pedidos por hora del día")
axes[0].set_xlabel("Hora"); axes[0].set_ylabel("% de pedidos")
axes[1].bar(range(7), dow_pct.reindex(range(7)).values, color=JITOMATE, width=0.7)
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(dias_nombre)
titulo_ax(axes[1], "Pedidos por día de la semana"); axes[1].set_ylabel("% de pedidos")
for ax in axes:
    ax.grid(axis="y"); ax.set_axisbelow(True)
guardar(fig, "02_ritmo"); plt.show()

hora_pico = int(horas_pct.idxmax())
R["hora_pico"] = hora_pico
R["hora_pico_pct"] = float(horas_pct.max())
R["dia_pico"] = dias_nombre[int(dow_pct.idxmax())]
R["dia_pico_pct"] = float(dow_pct.max())
print(f"Hora pico: {hora_pico}:00 ({pct(R['hora_pico_pct'])}) · Día pico: {R['dia_pico']} ({pct(R['dia_pico_pct'])})")

→ figs/02_ritmo.png
Hora pico: 18:00 (10.96%) · Día pico: Dom (16.78%)


In [7]:
# --- Dispersión de la matriz usuario × restaurante ---
pares = ped_tr.groupby(["customer_id", "vendor_id"], observed=True).size()
densidad = len(pares) / (R["usuarios_train"] * R["vendors"]) * 100
R["densidad_matriz"] = float(densidad)
R["pedidos_por_usuario"] = float(ped_tr.groupby("customer_id", observed=True).size().mean())
R["vendors_por_usuario"] = float(ped_tr.groupby("customer_id", observed=True)["vendor_id"]
                                 .nunique().mean())
print(f"Densidad de la matriz: {densidad:.4f}% (celdas con al menos un pedido)")
print(f"Promedio: {R['pedidos_por_usuario']:.2f} pedidos y {R['vendors_por_usuario']:.2f} restaurantes distintos por usuario")

Densidad de la matriz: 0.0364% (celdas con al menos un pedido)
Promedio: 3.59 pedidos y 2.70 restaurantes distintos por usuario


## 3. Tasa de repetición

La pregunta central del proyecto: cuando un usuario hace un pedido en el periodo
de prueba, ¿va a un restaurante que ya conocía o a uno nuevo?

In [8]:
hist = ped_tr.groupby("customer_id", observed=True)["vendor_id"].apply(set).to_dict()
te = ped_te[["customer_id", "vendor_id", "dia", "hora", "dow", "order_id"]].copy()
te["conocido"] = [v in hist.get(c, ()) for c, v in zip(te.customer_id, te.vendor_id)]
te["usuario_nuevo"] = [c not in hist for c in te.customer_id]

conocidos = te.loc[~te.usuario_nuevo, "conocido"]
R["tasa_repeticion"] = float(conocidos.mean() * 100)
R["pedidos_test_usuarios_conocidos"] = int(len(conocidos))
R["pct_pedidos_usuario_nuevo"] = float(te.usuario_nuevo.mean() * 100)
print(f"Tasa de repetición: {pct(R['tasa_repeticion'])} de los pedidos de prueba van a un restaurante ya conocido")
print(f"Arranque en frío: {pct(R['pct_pedidos_usuario_nuevo'])} de los pedidos de prueba son de usuarios sin historial")

# Tasa de repetición por número de pedidos previos
n_prev = ped_tr.groupby("customer_id", observed=True).size()
te_c = te.loc[~te.usuario_nuevo].copy()
te_c["n_prev"] = te_c.customer_id.map(n_prev).astype(float)
cortes = [0, 1, 2, 4, 9, 19, np.inf]
etiquetas = ["1", "2", "3-4", "5-9", "10-19", "20+"]
te_c["segmento"] = pd.cut(te_c.n_prev, bins=cortes, labels=etiquetas)
rep_seg = te_c.groupby("segmento", observed=True)["conocido"].mean() * 100
R["repeticion_por_segmento"] = {str(k): float(v) for k, v in rep_seg.items()}

fig, ax = plt.subplots(figsize=(6.2, 3.4))
ax.bar(range(len(rep_seg)), rep_seg.values, color=JITOMATE, width=0.7)
for i, v in enumerate(rep_seg.values):
    ax.text(i, v + 1.5, pct(v), ha="center", color=CARBON, fontsize=9, fontweight="bold")
ax.set_xticks(range(len(rep_seg))); ax.set_xticklabels(rep_seg.index)
ax.set_xlabel("Pedidos previos del usuario"); ax.set_ylabel("Pedidos a un lugar conocido (%)")
ax.set_ylim(0, min(100, rep_seg.max() * 1.25))
titulo_ax(ax, "Entre más pide un usuario, más repite")
ax.grid(axis="y"); ax.set_axisbelow(True)
guardar(fig, "03_repeticion"); plt.show()

Tasa de repetición: 37.06% de los pedidos de prueba van a un restaurante ya conocido
Arranque en frío: 15.20% de los pedidos de prueba son de usuarios sin historial


→ figs/03_repeticion.png


## 4. Minería de elementos frecuentes

Tres cosas: **A Priori vs FP-Growth** sobre las canastas de cocinas por usuario
(para comparar costo), reglas de **producto dentro del pedido** con una matriz
dispersa de co-ocurrencia, y **PrefixSpan** sobre la secuencia de cocinas.

In [9]:
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

cocina = vendors.set_index("vendor_id")["primary_cuisine"]
ped_tr["cocina"] = ped_tr.vendor_id.astype(str).map(cocina).astype("category")
ped_te["cocina"] = ped_te.vendor_id.astype(str).map(cocina).astype("category")

canastas = (ped_tr.loc[ped_tr.cocina.notna()]
            .groupby(["customer_id", "cocina"], observed=True).size().unstack(fill_value=0) > 0)
if SMOKE:
    canastas = canastas.sample(min(len(canastas), 50_000), random_state=SEED)
canastas = canastas.loc[canastas.sum(axis=1) >= 2]     # usuarios con 2+ cocinas
print("Canastas de cocinas:", miles(len(canastas)), "usuarios ×", canastas.shape[1], "cocinas")

MIN_SUP = 0.02
t0 = time.time(); frec_fp = fpgrowth(canastas, min_support=MIN_SUP, use_colnames=True); t_fp = time.time() - t0
t0 = time.time(); frec_ap = apriori(canastas, min_support=MIN_SUP, use_colnames=True); t_ap = time.time() - t0
R["fpgrowth_segundos"] = float(t_fp); R["apriori_segundos"] = float(t_ap)
R["itemsets_frecuentes"] = int(len(frec_fp))
print(f"FP-Growth: {t_fp:.2f} s · A Priori: {t_ap:.2f} s · {len(frec_fp)} itemsets (soporte ≥ {MIN_SUP:.0%})")

reglas = association_rules(frec_fp, metric="lift", min_threshold=1.2)
reglas = reglas.sort_values("lift", ascending=False)
reglas["antecedente"] = reglas.antecedents.apply(lambda s: ", ".join(sorted(s)))
reglas["consecuente"] = reglas.consequents.apply(lambda s: ", ".join(sorted(s)))
top_reglas = reglas[["antecedente", "consecuente", "support", "confidence", "lift"]].head(10)
R["reglas_cocina"] = [
    {"antecedente": a, "consecuente": c, "soporte": float(s) * 100,
     "confianza": float(cf) * 100, "lift": float(l)}
    for a, c, s, cf, l in top_reglas.itertuples(index=False, name=None)]
top_reglas

Canastas de cocinas: 247,237 usuarios × 78 cocinas


FP-Growth: 2.29 s · A Priori: 1.55 s · 104 itemsets (soporte ≥ 2%)


,antecedente,consecuente,support,confidence,lift
49,noodles,dim sum,0.020721,0.125465,1.768409
48,dim sum,noodles,0.020721,0.292059,1.768409
24,noodles,soups,0.024709,0.149613,1.675646
25,soups,noodles,0.024709,0.276738,1.675646
47,dim sum,chinese,0.029470,0.415370,1.564253
46,chinese,dim sum,0.029470,0.110981,1.564253
32,noodles,rice dishes,0.026513,0.160536,1.541495
33,rice dishes,noodles,0.026513,0.254583,1.541495
26,chinese,soups,0.036362,0.136936,1.533667
27,soups,chinese,0.036362,0.407248,1.533667


In [10]:
# --- Reglas de producto dentro del pedido (el caso "¿le agregas algo más?") ---
multi = lineas_tr.loc[lineas_tr.order_id.isin(ped_tr.loc[ped_tr.n_productos >= 2, "order_id"])]
ord_codes, ord_uniq = pd.factorize(multi.order_id)
prod_codes, prod_uniq = pd.factorize(multi.product_id.astype(str))
A = sp.csr_matrix((np.ones(len(multi), dtype=np.float32), (ord_codes, prod_codes)),
                  shape=(len(ord_uniq), len(prod_uniq)))
A.data[:] = 1.0
co = sp.triu(A.T @ A, k=1).tocoo()
conteo_prod = np.asarray(A.sum(axis=0)).ravel()
n_ped_multi = A.shape[0]

mask = co.data >= (5 if SMOKE else 30)
pares_df = pd.DataFrame({
    "ia": co.row[mask], "ib": co.col[mask], "juntos": co.data[mask]})
pares_df["a"] = prod_uniq[pares_df.ia.to_numpy()]
pares_df["b"] = prod_uniq[pares_df.ib.to_numpy()]
pares_df["soporte"] = pares_df.juntos / n_ped_multi * 100
pares_df["confianza"] = pares_df.juntos / conteo_prod[pares_df.ia.to_numpy()] * 100
nombre_prod = products.drop_duplicates("product_id").set_index("product_id")["name"]
pares_df["producto A"] = pares_df.a.map(nombre_prod)
pares_df["producto B"] = pares_df.b.map(nombre_prod)
top_pares = pares_df.sort_values("juntos", ascending=False).head(10)
R["pares_producto"] = [
    {"a": str(r["producto A"]), "b": str(r["producto B"]), "juntos": int(r.juntos),
     "confianza": float(r.confianza)} for _, r in top_pares.iterrows()]
print("Pares de productos que más se piden juntos:")
top_pares[["producto A", "producto B", "juntos", "confianza"]]

Pares de productos que más se piden juntos:


,producto A,producto B,juntos,confianza
3917,Claypot Bak Kut Teh 阿嬷肉骨茶,Rice 白饭,293.0,83.714287
2842,Rice,Claypot Bak Kut Teh Original,261.0,65.087280
3276,Youtiao Rojak,Black Rojak,236.0,67.621780
77,White Rice 白饭,Regular,232.0,56.038647
1119,6. Sesame Chicken 砂锅麻油宫保鸡丁,Plain Rice,210.0,81.081078
1658,C1s. Pork Ribs Soup,A1s. Fragrant White Rice,210.0,84.000000
1082,Rice,Claypot Bak Kut Teh Original,197.0,51.706036
3278,Plain Rice,6. Sesame Chicken 砂锅麻油宫保鸡丁,193.0,51.193638
91,Otah 窝咑,Small 小,193.0,67.013893
812,10. Prawn Dumpling 虾饺,09. Siew Mai 烧卖,190.0,62.295078


In [11]:
# --- PrefixSpan sobre la secuencia de cocinas de cada usuario ---
def prefixspan(secuencias, min_sup, max_len=3):
    """PrefixSpan clásico sobre secuencias de símbolos (items simples)."""
    resultados = []

    def recurre(prefijo, proyectada):
        # conteo de símbolos que pueden extender el prefijo
        conteo = {}
        for idx, pos in proyectada:
            vistos = set()
            for s in secuencias[idx][pos:]:
                if s not in vistos:
                    vistos.add(s)
                    conteo[s] = conteo.get(s, 0) + 1
        for simbolo, c in conteo.items():
            if c < min_sup:
                continue
            nuevo = prefijo + [simbolo]
            resultados.append((tuple(nuevo), c))
            if len(nuevo) < max_len:
                nueva_proj = []
                for idx, pos in proyectada:
                    seq = secuencias[idx]
                    for j in range(pos, len(seq)):
                        if seq[j] == simbolo:
                            nueva_proj.append((idx, j + 1))
                            break
                recurre(nuevo, nueva_proj)

    recurre([], [(i, 0) for i in range(len(secuencias))])
    return resultados

seq_src = ped_tr.loc[ped_tr.cocina.notna(), ["customer_id", "dia", "order_id", "cocina"]]
seq_src = seq_src.sort_values(["customer_id", "dia", "order_id"])
tam = seq_src.groupby("customer_id", observed=True).size()
elegibles = tam[tam.between(3, 30)].index
elegibles = pd.Index(rng.choice(np.asarray(elegibles), size=min(N_SEQ_USERS, len(elegibles)),
                                replace=False))
sub = seq_src[seq_src.customer_id.isin(elegibles)]
secuencias = [list(g) for _, g in sub.groupby("customer_id", observed=True)["cocina"]]
print("Secuencias:", miles(len(secuencias)), "· largo medio:", f"{np.mean([len(s) for s in secuencias]):.1f}")

min_sup = int(0.02 * len(secuencias))
t0 = time.time()
patrones = prefixspan(secuencias, min_sup=min_sup, max_len=3)
print(f"PrefixSpan en {time.time()-t0:.1f} s · {len(patrones)} patrones con soporte ≥ 2%")

pat_df = pd.DataFrame([{"patrón": " → ".join(p), "largo": len(p),
                        "soporte": c / len(secuencias) * 100} for p, c in patrones])
top_sec = pat_df[pat_df.largo >= 2].sort_values("soporte", ascending=False).head(10)
R["patrones_secuencia"] = [{"patron": r["patrón"], "soporte": float(r.soporte)}
                           for _, r in top_sec.iterrows()]
top_sec

Secuencias: 30,000 · largo medio: 6.6


PrefixSpan en 0.9 s · 128 patrones con soporte ≥ 2%


,patrón,largo,soporte
14,chinese → chinese,2,14.040000
110,indian → indian,2,7.916667
16,chinese → chinese → chinese,3,7.286667
45,noodles → noodles,2,7.163333
60,coffee → coffee,2,6.223333
83,bubble tea → bubble tea,2,6.220000
12,chinese → noodles,2,5.533333
42,noodles → chinese,2,5.460000
32,japanese → japanese,2,5.283333
69,dessert → dessert,2,5.273333


## 5. Análisis de vínculos

Construimos el grafo bipartito usuario–restaurante y lo proyectamos a un grafo
restaurante–restaurante (dos restaurantes se conectan si comparten clientes).
Sobre él corremos **PageRank con el método de potencias** (implementado a mano),
**HITS** y **caminatas aleatorias con reinicio** como recomendador.

In [12]:
usuarios = pd.Index(ped_tr.customer_id.astype(str).unique())
vendor_idx = pd.Index(vendors.vendor_id.astype(str).unique())
u_pos = pd.Series(np.arange(len(usuarios)), index=usuarios)
v_pos = pd.Series(np.arange(len(vendor_idx)), index=vendor_idx)

ui = ped_tr.customer_id.astype(str).map(u_pos).to_numpy()
vi = ped_tr.vendor_id.astype(str).map(v_pos).to_numpy()
ok = ~np.isnan(vi)
ui, vi = ui[ok].astype(np.int32), vi[ok].astype(np.int32)

M = sp.csr_matrix((np.ones(len(ui), dtype=np.float32), (ui, vi)),
                  shape=(len(usuarios), len(vendor_idx)))
M.sum_duplicates()
print("Matriz usuario × restaurante:", M.shape, "·", miles(M.nnz), "celdas ocupadas")

# Proyección restaurante–restaurante (clientes compartidos), sin la diagonal
B = M.copy(); B.data[:] = 1.0
S = (B.T @ B).tocsr()
S.setdiag(0); S.eliminate_zeros()
MIN_CLIENTES_COMPARTIDOS = 2      # une dos restaurantes solo si comparten 2+ clientes
S.data[S.data < MIN_CLIENTES_COMPARTIDOS] = 0
S.eliminate_zeros()
print("Grafo de restaurantes:", S.shape[0], "nodos ·", miles(S.nnz // 2), "aristas",
      f"(≥ {MIN_CLIENTES_COMPARTIDOS} clientes compartidos)")

Matriz usuario × restaurante: (476150, 7411) · 1,285,976 celdas ocupadas


Grafo de restaurantes: 7411 nodos · 498,014 aristas (≥ 2 clientes compartidos)


In [13]:
def normaliza_columnas(X):
    """Matriz de transición columna-estocástica."""
    grados = np.asarray(X.sum(axis=0)).ravel()
    inv = np.zeros_like(grados); nz = grados > 0
    inv[nz] = 1.0 / grados[nz]
    return X @ sp.diags(inv), nz

def pagerank_potencias(X, d=0.85, tol=1e-10, max_iter=200):
    """PageRank por el método de potencias (tema 1.1 aplicado al 3.1)."""
    n = X.shape[0]
    P, tiene_grado = normaliza_columnas(X)
    r = np.full(n, 1.0 / n)
    sin_grado = ~tiene_grado
    for it in range(max_iter):
        nuevo = d * (P @ r) + d * r[sin_grado].sum() / n + (1 - d) / n
        delta = np.abs(nuevo - r).sum()
        r = nuevo
        if delta < tol:
            break
    return r, it + 1, delta

t0 = time.time()
pr, iters, delta = pagerank_potencias(S)
R["pagerank_iteraciones"] = int(iters)
R["pagerank_segundos"] = float(time.time() - t0)
print(f"PageRank convergió en {iters} iteraciones ({R['pagerank_segundos']:.1f} s, delta={delta:.2e})")

pop = np.asarray(M.sum(axis=0)).ravel()
tabla_pr = pd.DataFrame({
    "vendor_id": vendor_idx, "pagerank": pr, "pedidos": pop,
    "cocina": pd.Series(vendor_idx).map(vendors.set_index("vendor_id")["primary_cuisine"]).values,
}).sort_values("pagerank", ascending=False)
R["top_pagerank"] = [{"vendor": r.vendor_id, "cocina": str(r.cocina), "pedidos": int(r.pedidos)}
                     for _, r in tabla_pr.head(10).iterrows()]

# Correlación entre centralidad y popularidad: ¿PageRank dice algo nuevo?
from scipy.stats import spearmanr
rho = spearmanr(pr, pop).statistic
R["correlacion_pagerank_popularidad"] = float(rho)
print(f"Correlación de Spearman entre PageRank y número de pedidos: {rho:.3f}")
tabla_pr.head(10)

PageRank convergió en 95 iteraciones (0.2 s, delta=8.32e-11)


Correlación de Spearman entre PageRank y número de pedidos: 0.957


,vendor_id,pagerank,pedidos,cocina
5142,f3024b8e,0.000725,571.0,dim sum
5507,e35a8c91,0.000691,559.0,japanese
578,89e1250f,0.000681,543.0,thai
6909,22c887de,0.000679,540.0,singaporean
5211,930068e0,0.000670,594.0,japanese
5166,0033029f,0.000665,560.0,dim sum
3309,53fb28d4,0.000645,548.0,chinese
5439,7f528535,0.000626,574.0,japanese
5342,1bd969b8,0.000625,529.0,japanese
495,e13dc36b,0.000625,573.0,porridge


In [14]:
# --- HITS sobre el grafo bipartito: usuarios = hubs, restaurantes = autoridades ---
def hits(X, max_iter=100, tol=1e-10):
    a = np.ones(X.shape[1]) / np.sqrt(X.shape[1])
    for _ in range(max_iter):
        h = X @ a; h /= np.linalg.norm(h) or 1
        nuevo = X.T @ h; nuevo /= np.linalg.norm(nuevo) or 1
        if np.abs(nuevo - a).sum() < tol:
            a = nuevo; break
        a = nuevo
    return h, a

hubs, autoridades = hits(M)
tabla_hits = pd.DataFrame({"vendor_id": vendor_idx, "autoridad": autoridades, "pedidos": pop})
tabla_hits["cocina"] = pd.Series(vendor_idx).map(
    vendors.set_index("vendor_id")["primary_cuisine"]).values
tabla_hits = tabla_hits.sort_values("autoridad", ascending=False)
R["top_autoridades"] = [{"vendor": r.vendor_id, "cocina": str(r.cocina), "pedidos": int(r.pedidos)}
                        for _, r in tabla_hits.head(10).iterrows()]
R["correlacion_hits_popularidad"] = float(spearmanr(autoridades, pop).statistic)
print(f"Correlación autoridad–popularidad: {R['correlacion_hits_popularidad']:.3f}")
tabla_hits.head(10)

Correlación autoridad–popularidad: 0.552


,vendor_id,autoridad,pedidos,cocina
697,e09f6517,0.998587,336.0,asian
4,0946c9e5,0.025557,291.0,bak kut teh
710,3c30529e,0.022114,134.0,asian
738,95d54bdd,0.018739,523.0,asian
2353,f0b76186,0.015033,470.0,indian
1750,dc7f9f87,0.014887,171.0,coffee
7129,60257fe4,0.011838,538.0,chicken rice
3086,61081731,0.010203,555.0,chinese
5450,91704ac5,0.009456,358.0,japanese
7060,2963c19a,0.006539,532.0,chicken rice


## 6. Comunidades de gusto

Louvain sobre el grafo de restaurantes, quedándonos con las aristas más fuertes
de cada nodo para que el grafo no sea una masa densa.

In [15]:
import networkx as nx

K_VECINOS = 15
# nos quedamos con las K aristas más fuertes de cada restaurante
filas, cols, pesos = [], [], []
S_lil = S.tocsr()
for i in range(S_lil.shape[0]):
    ini, fin = S_lil.indptr[i], S_lil.indptr[i + 1]
    if fin == ini:
        continue
    vecinos = S_lil.indices[ini:fin]; w = S_lil.data[ini:fin]
    sel = np.argsort(-w)[:K_VECINOS]
    filas.append(np.full(len(sel), i)); cols.append(vecinos[sel]); pesos.append(w[sel])
ii, jj, ww = np.concatenate(filas), np.concatenate(cols), np.concatenate(pesos)
lo, hi = np.minimum(ii, jj), np.maximum(ii, jj)
aristas = (pd.DataFrame({"i": lo, "j": hi, "w": ww})
           .drop_duplicates(subset=["i", "j"]))
G = nx.Graph()
G.add_nodes_from(range(len(vendor_idx)))
G.add_weighted_edges_from(aristas[["i", "j", "w"]].itertuples(index=False, name=None))

t0 = time.time()
comunidades = nx.community.louvain_communities(G, weight="weight", seed=SEED)
comunidades = sorted(comunidades, key=len, reverse=True)
print(f"Louvain: {len(comunidades)} comunidades en {time.time()-t0:.1f} s")

com_de = np.full(len(vendor_idx), -1)
for c, nodos in enumerate(comunidades):
    for n in nodos:
        com_de[n] = c
R["n_comunidades"] = int(len(comunidades))
R["modularidad"] = float(nx.community.modularity(G, comunidades, weight="weight"))
print(f"Modularidad: {R['modularidad']:.3f}")

cocina_v = pd.Series(vendor_idx).map(vendors.set_index("vendor_id")["primary_cuisine"]).values
resumen_com = []
for c, nodos in enumerate(comunidades[:6]):
    idx = np.fromiter(nodos, dtype=int)
    cocinas_c = pd.Series(cocina_v[idx]).value_counts().head(3)
    zonas = pd.Series(vendors.set_index("vendor_id")
                      .reindex(vendor_idx[idx])["geohash"].values).value_counts().head(2)
    resumen_com.append({"comunidad": c, "restaurantes": int(len(idx)),
                        "pedidos": int(pop[idx].sum()),
                        "cocinas": ", ".join(f"{k} ({v})" for k, v in cocinas_c.items()),
                        "zonas": ", ".join(str(k) for k in zonas.index)})
R["comunidades"] = resumen_com
pd.DataFrame(resumen_com)

Louvain: 568 comunidades en 1.2 s
Modularidad: 0.866


,comunidad,restaurantes,pedidos,cocinas,zonas
0,0,1620,260919,"chinese (159), indian (135), japanese (116)","w21z7, w21z6"
1,1,823,265178,"chinese (90), noodles (71), bubble tea (47)","w21zu, w21zg"
2,2,572,152397,"chinese (53), bubble tea (45), indian (31)","w21zt, w21zv"
3,3,555,135078,"chinese (52), noodles (42), indian (36)","w21zk, w21zs"
4,4,554,121826,"chinese (61), noodles (31), western (30)","w21ze, w21z7"
5,5,452,107048,"chinese (48), bubble tea (27), dessert (26)","w21z8, w21z9"


## 7. Modelos de recomendación y evaluación

Protocolo: para cada usuario de prueba, el sistema entrega **10 restaurantes** y
se compara contra los restaurantes a los que realmente pidió en el periodo de
prueba. Se reportan precision@10, recall@10, NDCG@10 y cobertura de catálogo.
Las recomendaciones **no** excluyen lo ya pedido, porque repetir es parte del
comportamiento real.

In [16]:
verdad = (ped_te.groupby("customer_id", observed=True)["vendor_id"]
          .apply(lambda s: set(s.astype(str))))
usuarios_eval = np.array([u for u in verdad.index.astype(str) if u in u_pos.index])
usuarios_eval = rng.permutation(usuarios_eval)[:N_EVAL_USERS]
eval_pos = u_pos.loc[usuarios_eval].to_numpy()
verdad_idx = []
for u in usuarios_eval:
    objetivo = {v_pos[v] for v in verdad[u] if v in v_pos.index}
    verdad_idx.append(objetivo)
print("Usuarios evaluados:", miles(len(usuarios_eval)))

# ¿Cuántos de sus pedidos de prueba son repetición?
repite_eval = np.array([len(verdad_idx[i] & set(M[eval_pos[i]].indices)) > 0
                        for i in range(len(usuarios_eval))])
R["usuarios_eval"] = int(len(usuarios_eval))
R["usuarios_eval_con_repeticion"] = float(repite_eval.mean() * 100)

Usuarios evaluados: 20,000


In [17]:
def ndcg_en_k(recomendados, relevantes, k=TOP_K):
    dcg = sum(1.0 / np.log2(i + 2) for i, v in enumerate(recomendados[:k]) if v in relevantes)
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevantes), k)))
    return dcg / ideal if ideal > 0 else 0.0

conocidos_idx = [set(M[p].indices.tolist()) for p in eval_pos]   # lo que ya pidió cada usuario

def evalua(nombre, scores_fn, k=TOP_K, lotes=2_000):
    """scores_fn(posiciones_usuarios) -> matriz densa (n_usuarios × n_vendors)."""
    precs, recs, ndcgs = [], [], []
    ac_rep = tot_rep = ac_nue = tot_nue = 0
    catalogo = set()
    t0 = time.time()
    for ini in range(0, len(eval_pos), lotes):
        fin = min(ini + lotes, len(eval_pos))
        S_lote = scores_fn(eval_pos[ini:fin])
        top = np.argpartition(-S_lote, k, axis=1)[:, :k]
        filas = np.arange(top.shape[0])[:, None]
        top = top[filas, np.argsort(-S_lote[filas, top], axis=1)]
        for f, i in enumerate(range(ini, fin)):
            rel = verdad_idx[i]
            rec = top[f]
            catalogo.update(rec.tolist())
            aciertos = sum(1 for v in rec if v in rel)
            precs.append(aciertos / k)
            recs.append(aciertos / len(rel) if rel else 0.0)
            ndcgs.append(ndcg_en_k(list(rec), rel, k))
            # desglose: ¿acertó en lugares que ya conocía o en lugares nuevos?
            rep = rel & conocidos_idx[i]
            nue = rel - conocidos_idx[i]
            recomendados = set(rec.tolist())
            ac_rep += len(recomendados & rep); tot_rep += len(rep)
            ac_nue += len(recomendados & nue); tot_nue += len(nue)
    return {"modelo": nombre,
            "precision@10": float(np.mean(precs) * 100),
            "recall@10": float(np.mean(recs) * 100),
            "ndcg@10": float(np.mean(ndcgs) * 100),
            "recall repetidos": float(ac_rep / tot_rep * 100) if tot_rep else 0.0,
            "recall nuevos": float(ac_nue / tot_nue * 100) if tot_nue else 0.0,
            "cobertura": float(len(catalogo) / len(vendor_idx) * 100),
            "segundos": float(time.time() - t0)}

resultados = []

In [18]:
# --- Modelo 1: popularidad global (todo el periodo de entrenamiento) ---
pop_scores = pop.astype(np.float32)
resultados.append(evalua("Popularidad", lambda idx: np.tile(pop_scores, (len(idx), 1))))
print(resultados[-1])

# Hallazgo: los restaurantes más pedidos del periodo completo pueden haber cerrado.
top20 = np.argsort(-pop)[:20]
vendors_test = set(ped_te.vendor_id.astype(str).unique())
sin_test = [str(vendor_idx[i]) for i in top20 if str(vendor_idx[i]) not in vendors_test]
R["top20_populares_sin_pedidos_en_test"] = int(len(sin_test))
print(f"{len(sin_test)} de los 20 restaurantes más pedidos del entrenamiento no reciben "
      f"ni un pedido en el periodo de prueba")

# --- Modelo 1b: popularidad reciente (últimos 14 días del entrenamiento) ---
VENTANA = 14
recientes = ped_tr[ped_tr.dia > ped_tr.dia.max() - VENTANA]
pop_rec = np.zeros(len(vendor_idx), dtype=np.float32)
cuentas = recientes.vendor_id.astype(str).value_counts()
for v, c in cuentas.items():
    if v in v_pos.index:
        pop_rec[v_pos[v]] = c
resultados.append(evalua(f"Popularidad reciente ({VENTANA} días)",
                         lambda idx: np.tile(pop_rec, (len(idx), 1))))
print(resultados[-1])

{'modelo': 'Popularidad', 'precision@10': 0.0, 'recall@10': 0.0, 'ndcg@10': 0.0, 'recall repetidos': 0.0, 'recall nuevos': 0.0, 'cobertura': 0.1349345567399811, 'segundos': 1.1948883533477783}
8 de los 20 restaurantes más pedidos del entrenamiento no reciben ni un pedido en el periodo de prueba


{'modelo': 'Popularidad reciente (14 días)', 'precision@10': 0.117, 'recall@10': 0.6889505772005773, 'ndcg@10': 0.39128876010023045, 'recall repetidos': 0.41720749119228634, 'recall nuevos': 0.8664160630787567, 'cobertura': 0.1349345567399811, 'segundos': 1.1069095134735107}


In [19]:
# --- Modelo 2: "lo de siempre" (los restaurantes del propio usuario) ---
M_csr = M.tocsr().astype(np.float32)
def scores_repite(idx):
    base = M_csr[idx].toarray()
    return base * 1000.0 + pop_scores / (pop_scores.max() + 1)   # desempate por popularidad
resultados.append(evalua("Lo de siempre", scores_repite))
print(resultados[-1])

{'modelo': 'Lo de siempre', 'precision@10': 5.1715, 'recall@10': 31.10559379509379, 'ndcg@10': 26.29266499522267, 'recall repetidos': 95.89282403115149, 'recall nuevos': 0.0, 'cobertura': 85.696936985562, 'segundos': 1.4706120491027832}


In [20]:
# --- Modelo 3: basado en contenido (cocina, zona y precio típico) ---
precio_vendor = (products.groupby("vendor_id")["unit_price"].mean()
                 .reindex(vendor_idx).fillna(products.unit_price.mean()))
v_meta = pd.DataFrame({"vendor_id": vendor_idx})
v_meta["cocina"] = cocina_v
v_meta["zona"] = vendors.set_index("vendor_id").reindex(vendor_idx)["geohash"].values
F_cocina = pd.get_dummies(v_meta.cocina.fillna("desconocida"), dtype=np.float32)
F_zona = pd.get_dummies(v_meta.zona.fillna("na").astype(str).str.slice(0, 5), dtype=np.float32)
precio_norm = ((precio_vendor - precio_vendor.mean()) / (precio_vendor.std() + 1e-9)).to_numpy()
F = np.hstack([F_cocina.to_numpy(), F_zona.to_numpy() * 0.7,
               precio_norm.reshape(-1, 1).astype(np.float32) * 0.3])
F = F / (np.linalg.norm(F, axis=1, keepdims=True) + 1e-9)
F = F.astype(np.float32)

def scores_contenido(idx):
    perfil = M_csr[idx] @ F                      # perfil = promedio de lo que pidió
    perfil /= (np.linalg.norm(perfil, axis=1, keepdims=True) + 1e-9)
    return perfil @ F.T
resultados.append(evalua("Contenido", scores_contenido))
print(resultados[-1])

{'modelo': 'Contenido', 'precision@10': 2.5860000000000003, 'recall@10': 17.990750582750582, 'ndcg@10': 12.655450675224392, 'recall repetidos': 42.12868533283887, 'recall nuevos': 2.878885119647933, 'cobertura': 99.43327486169208, 'segundos': 1.3153419494628906}


In [21]:
# --- Modelo 4: filtrado colaborativo ítem–ítem (coseno) ---
Bc = M_csr.copy(); Bc.data[:] = 1.0
norma = np.sqrt(np.asarray(Bc.power(2).sum(axis=0)).ravel()) + 1e-9
Sim = (Bc.T @ Bc).toarray().astype(np.float32)
Sim /= norma[:, None]; Sim /= norma[None, :]
np.fill_diagonal(Sim, 0.0)
def scores_item_item(idx):
    return (M_csr[idx] @ Sim)
resultados.append(evalua("CF ítem-ítem", scores_item_item))
print(resultados[-1])

{'modelo': 'CF ítem-ítem', 'precision@10': 1.7690000000000003, 'recall@10': 9.924734515484515, 'ndcg@10': 5.671951596428283, 'recall repetidos': 12.729464120155756, 'recall nuevos': 9.924818923627027, 'cobertura': 93.01038996086898, 'segundos': 3.2102441787719727}


In [22]:
# --- Modelo 5: factores latentes con ALS implícito ---
import implicit
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
FACTORES = 32 if SMOKE else 64
modelo_als = implicit.als.AlternatingLeastSquares(
    factors=FACTORES, regularization=0.05, alpha=20.0,
    iterations=8 if SMOKE else 20, random_state=SEED, use_gpu=False)
t0 = time.time()
modelo_als.fit(M_csr)
R["als_segundos"] = float(time.time() - t0)
R["als_factores"] = FACTORES
print(f"ALS entrenado en {R['als_segundos']:.1f} s")

U = np.asarray(modelo_als.user_factors, dtype=np.float32)
V = np.asarray(modelo_als.item_factors, dtype=np.float32)
def scores_als(idx):
    return U[idx] @ V.T
resultados.append(evalua("ALS (factores latentes)", scores_als))
print(resultados[-1])

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.11/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

  5%|▌         | 1/20 [00:02<00:47,  2.50s/it]

 10%|█         | 2/20 [00:04<00:34,  1.94s/it]

 15%|█▌        | 3/20 [00:05<00:29,  1.75s/it]

 20%|██        | 4/20 [00:07<00:26,  1.63s/it]

... (15 outputs left out)

100%|██████████| 20/20 [00:34<00:00,  1.62s/it]

100%|██████████| 20/20 [00:34<00:00,  1.73s/it]

ALS entrenado en 34.8 s


{'modelo': 'ALS (factores latentes)', 'precision@10': 3.6410000000000005, 'recall@10': 24.00878375790876, 'ndcg@10': 16.11774857229858, 'recall repetidos': 48.25699981457445, 'recall nuevos': 9.521408269918402, 'cobertura': 36.526784509512886, 'segundos': 1.2931687831878662}


In [23]:
# --- Modelo 6: SVD truncado sobre la matriz dispersa ---
from scipy.sparse.linalg import svds
k_svd = 32 if SMOKE else 64
Us, ss, Vts = svds(M_csr.astype(np.float64), k=k_svd)
Us, ss, Vts = Us.astype(np.float32), ss.astype(np.float32), Vts.astype(np.float32)
def scores_svd(idx):
    return (Us[idx] * ss) @ Vts
resultados.append(evalua("SVD truncado", scores_svd))
print(resultados[-1])

{'modelo': 'SVD truncado', 'precision@10': 1.0574999999999999, 'recall@10': 6.106433857808858, 'ndcg@10': 3.5917849635170076, 'recall repetidos': 12.18245874281476, 'recall nuevos': 3.6719537911433022, 'cobertura': 8.676291998380785, 'segundos': 1.2156832218170166}


In [24]:
# --- Modelo 7: caminata aleatoria con reinicio (PageRank personalizado) ---
# se camina solo por las aristas más fuertes de cada restaurante (las del grafo de Louvain)
S_walk = sp.csr_matrix(
    (np.concatenate(pesos), (np.concatenate(filas), np.concatenate(cols))), shape=S.shape)
S_walk = S_walk.maximum(S_walk.T)                # simetriza
P_walk, _ = normaliza_columnas(S_walk)
P_walk = P_walk.T.tocsr().astype(np.float32)     # fila i = distribución desde i
print("Aristas usadas en la caminata:", miles(P_walk.nnz))
ALFA = 0.6
def scores_ppr(idx, pasos=6):
    p0 = M_csr[idx].toarray()
    p0 /= (p0.sum(axis=1, keepdims=True) + 1e-9)
    x = p0.copy()
    for _ in range(pasos):
        x = ALFA * (x @ P_walk) + (1 - ALFA) * p0
    return x
resultados.append(evalua("Caminata aleatoria (PPR)", scores_ppr, lotes=1_000))
print(resultados[-1])

Aristas usadas en la caminata: 159,714


{'modelo': 'Caminata aleatoria (PPR)', 'precision@10': 5.900499999999999, 'recall@10': 36.25335028860028, 'ndcg@10': 28.746308054108237, 'recall repetidos': 96.48618579640275, 'recall nuevos': 6.390391491702577, 'cobertura': 86.06126028875995, 'segundos': 12.070566415786743}


In [25]:
tabla = pd.DataFrame(resultados).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
R["tabla_modelos"] = tabla.to_dict("records")
tabla.round(2)

,modelo,precision@10,recall@10,ndcg@10,recall repetidos,recall nuevos,cobertura,segundos
0,Caminata aleatoria (PPR),5.90,36.25,28.75,96.49,6.39,86.06,12.07
1,Lo de siempre,5.17,31.11,26.29,95.89,0.00,85.70,1.47
2,ALS (factores latentes),3.64,24.01,16.12,48.26,9.52,36.53,1.29
3,Contenido,2.59,17.99,12.66,42.13,2.88,99.43,1.32
4,CF ítem-ítem,1.77,9.92,5.67,12.73,9.92,93.01,3.21
5,SVD truncado,1.06,6.11,3.59,12.18,3.67,8.68,1.22
6,Popularidad reciente (14 días),0.12,0.69,0.39,0.42,0.87,0.13,1.11
7,Popularidad,0.00,0.00,0.00,0.00,0.00,0.13,1.19


## 8. Capa "repetir o explorar"

Un clasificador decide, por usuario, si su próximo pedido será a un lugar
conocido. Con esa probabilidad se mezclan "lo de siempre" y el modelo de
descubrimiento (ALS). Las variables son de comportamiento, no identificadores,
así que el modelo se puede transferir a otro negocio.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

CORTE = int(ped_tr.dia.max() * 0.8)      # últimas semanas del train = ventana de etiqueta
hist_a = ped_tr[ped_tr.dia <= CORTE]
hist_b = ped_tr[ped_tr.dia > CORTE]

def variables(hist, usuarios_obj, dia_ref):
    h = hist[["customer_id", "vendor_id", "dia", "hora", "dow"]].copy()
    h["finde"] = (h["dow"] >= 5).astype(np.int8)
    g = h.groupby("customer_id", observed=True)
    agg = g.agg(n_pedidos=("vendor_id", "size"), n_vendors=("vendor_id", "nunique"),
                ultimo=("dia", "max"), hora_media=("hora", "mean"),
                prop_finde=("finde", "mean"))
    agg.index = agg.index.astype(str)
    agg = agg.reindex(usuarios_obj)
    agg["n_pedidos"] = agg["n_pedidos"].fillna(0)
    agg["n_vendors"] = agg["n_vendors"].fillna(0)
    agg["ultimo"] = agg["ultimo"].fillna(-1)
    agg["hora_media"] = agg["hora_media"].fillna(12)
    agg["prop_finde"] = agg["prop_finde"].fillna(0)
    X = pd.DataFrame({
        "n_pedidos": agg.n_pedidos,
        "n_vendors": agg.n_vendors,
        "dias_desde_ultimo": dia_ref - agg.ultimo,
        "hora_media": agg.hora_media,
        "prop_finde": agg.prop_finde,
        "tasa_repeticion_previa": ((agg.n_pedidos - agg.n_vendors) /
                                   agg.n_pedidos.replace(0, np.nan)).fillna(0),
        "pedidos_por_vendor": (agg.n_pedidos / agg.n_vendors.replace(0, np.nan)).fillna(0),
    })
    return X.fillna(0)

usuarios_b = hist_b.customer_id.astype(str).unique()
X_train = variables(hist_a, usuarios_b, CORTE)
primer_b = (hist_b.sort_values(["customer_id", "dia"])
            .groupby("customer_id", observed=True).first())
hist_a_sets = hist_a.groupby("customer_id", observed=True)["vendor_id"].apply(
    lambda s: set(s.astype(str)))
hist_a_sets.index = hist_a_sets.index.astype(str)
hist_a_dict = hist_a_sets.to_dict()
y_train = pd.Series(
    [1 if v in hist_a_dict.get(c, ()) else 0
     for c, v in zip(primer_b.index.astype(str), primer_b.vendor_id.astype(str))],
    index=primer_b.index.astype(str)).reindex(usuarios_b).fillna(0).astype(int)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

X_eval = variables(ped_tr, pd.Index(usuarios_eval), ped_tr.dia.max())
p_repite = clf.predict_proba(X_eval)[:, 1]
y_eval = np.array([1 if len(verdad_idx[i] & set(M_csr[eval_pos[i]].indices)) > 0 else 0
                   for i in range(len(usuarios_eval))])
R["auc_repetir"] = float(roc_auc_score(y_eval, p_repite))
R["coeficientes_repetir"] = dict(zip(X_train.columns, [float(c) for c in clf.coef_[0]]))
print(f"AUC del clasificador repetir/explorar: {R['auc_repetir']:.3f}")
pd.Series(R["coeficientes_repetir"]).sort_values(ascending=False)

AUC del clasificador repetir/explorar: 0.757


pedidos_por_vendor        1.311350
tasa_repeticion_previa    0.271683
prop_finde                0.239271
n_vendors                 0.151168
hora_media                0.009035
dias_desde_ultimo        -0.024519
n_pedidos                -0.064231
dtype: float64

In [27]:
# --- Híbrido: mezcla según la probabilidad de repetir ---
def normaliza_filas(X):
    mx = X.max(axis=1, keepdims=True)
    mn = X.min(axis=1, keepdims=True)
    return (X - mn) / (mx - mn + 1e-9)

pos_a_prob = {int(p): float(pr_) for p, pr_ in zip(eval_pos, p_repite)}

# el componente de descubrimiento es el mejor modelo puro según NDCG@10
puros = pd.DataFrame(resultados)
descubridores = {"ALS (factores latentes)": scores_als,
                 "Caminata aleatoria (PPR)": scores_ppr,
                 "CF ítem-ítem": scores_item_item,
                 "Contenido": scores_contenido}
mejor_desc = (puros[puros.modelo.isin(descubridores)]
              .sort_values("ndcg@10", ascending=False).iloc[0]["modelo"])
scores_descubre = descubridores[mejor_desc]
print("Componente de descubrimiento:", mejor_desc)

def scores_hibrido(idx):
    w = np.array([pos_a_prob[int(p)] for p in idx], dtype=np.float32)[:, None]
    repite = normaliza_filas(M_csr[idx].toarray().astype(np.float32))
    descubre = normaliza_filas(np.asarray(scores_descubre(idx), dtype=np.float32))
    return w * repite + (1 - w) * descubre

R["componente_descubrimiento"] = mejor_desc
resultados.append(evalua("Híbrido repetir/explorar", scores_hibrido, lotes=1_000))
print(resultados[-1])

tabla = pd.DataFrame(resultados).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
R["tabla_modelos"] = tabla.to_dict("records")
tabla.round(2)

Componente de descubrimiento: Caminata aleatoria (PPR)


{'modelo': 'Híbrido repetir/explorar', 'precision@10': 5.9065, 'recall@10': 36.26467586580087, 'ndcg@10': 28.758289096599842, 'recall repetidos': 96.65306879287967, 'recall nuevos': 6.362886219858805, 'cobertura': 86.14222102280394, 'segundos': 15.273149728775024}


,modelo,precision@10,recall@10,ndcg@10,recall repetidos,recall nuevos,cobertura,segundos
0,Híbrido repetir/explorar,5.91,36.26,28.76,96.65,6.36,86.14,15.27
1,Caminata aleatoria (PPR),5.90,36.25,28.75,96.49,6.39,86.06,12.07
2,Lo de siempre,5.17,31.11,26.29,95.89,0.00,85.70,1.47
3,ALS (factores latentes),3.64,24.01,16.12,48.26,9.52,36.53,1.29
4,Contenido,2.59,17.99,12.66,42.13,2.88,99.43,1.32
5,CF ítem-ítem,1.77,9.92,5.67,12.73,9.92,93.01,3.21
6,SVD truncado,1.06,6.11,3.59,12.18,3.67,8.68,1.22
7,Popularidad reciente (14 días),0.12,0.69,0.39,0.42,0.87,0.13,1.11
8,Popularidad,0.00,0.00,0.00,0.00,0.00,0.13,1.19


In [28]:
ganador = tabla.iloc[0]["modelo"]
ndcg_ganador = float(tabla.iloc[0]["ndcg@10"])
base_pop = float(tabla.loc[tabla.modelo.str.startswith("Popularidad"), "ndcg@10"].max())
base_siempre = float(tabla.loc[tabla.modelo == "Lo de siempre", "ndcg@10"].iloc[0])
R["modelo_ganador"] = ganador
R["ndcg_ganador"] = ndcg_ganador
R["ndcg_popularidad"] = base_pop
R["ndcg_lo_de_siempre"] = base_siempre
R["mejora_sobre_popularidad"] = float(ndcg_ganador / base_pop) if base_pop > 0.05 else None
R["mejora_sobre_lo_de_siempre"] = float((ndcg_ganador / base_siempre - 1) * 100)
print(f"Gana {ganador} con {ndcg_ganador:.2f} de NDCG@10")
print(f"  vs. mejor línea base de popularidad ({base_pop:.2f}): "
      f"{'×' + format(R['mejora_sobre_popularidad'], '.1f') if R['mejora_sobre_popularidad'] else 'la popularidad no acierta nada'}")
print(f"  vs. 'lo de siempre' ({base_siempre:.2f}): {R['mejora_sobre_lo_de_siempre']:+.2f}%")

fig, ax = plt.subplots(figsize=(7.6, 4.0))
t = tabla.sort_values("ndcg@10")
colores = [JITOMATE if m == ganador else COBALTO for m in t.modelo]
ax.barh(range(len(t)), t["ndcg@10"], color=colores, height=0.65)
for i, v in enumerate(t["ndcg@10"]):
    ax.text(v + 0.3, i, f"{v:.2f}", va="center", color=CARBON, fontsize=9, fontweight="bold")
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.modelo)
ax.set_xlabel("NDCG@10 (%)")
titulo_ax(ax, f"{ganador} es el mejor modelo (NDCG@10)")
ax.grid(axis="x"); ax.set_axisbelow(True)
guardar(fig, "04_modelos"); plt.show()

# --- Desglose: aciertos en lugares conocidos vs lugares nuevos ---
fig, ax = plt.subplots(figsize=(7.6, 4.0))
t2 = tabla.sort_values("recall@10")
y = np.arange(len(t2))
ax.barh(y - 0.19, t2["recall repetidos"], height=0.36, color=JITOMATE, label="Lugares conocidos")
ax.barh(y + 0.19, t2["recall nuevos"], height=0.36, color=COBALTO, label="Lugares nuevos")
ax.set_yticks(y); ax.set_yticklabels(t2.modelo)
ax.set_xlabel("Recall@10 (%)"); ax.legend(loc="lower right")
titulo_ax(ax, "Repetir es fácil; descubrir es el problema difícil")
ax.grid(axis="x"); ax.set_axisbelow(True)
guardar(fig, "07_repetir_vs_nuevos"); plt.show()

Gana Híbrido repetir/explorar con 28.76 de NDCG@10
  vs. mejor línea base de popularidad (0.39): ×73.5
  vs. 'lo de siempre' (26.29): +9.38%


→ figs/04_modelos.png


→ figs/07_repetir_vs_nuevos.png


## 9. Arranque en frío del negocio pequeño

Los restaurantes con poca historia son los que peor la pasan con el filtrado
colaborativo. Aquí medimos qué tan seguido cada modelo logra recomendar a un
restaurante **pequeño** (pocos pedidos en el entrenamiento) que el usuario
efectivamente visitó en el periodo de prueba.

In [29]:
UMBRAL_PEQUENO = 50 if not SMOKE else 20
pequenos = set(np.where(pop <= UMBRAL_PEQUENO)[0].tolist())
R["umbral_pequeno"] = UMBRAL_PEQUENO
R["n_vendors_pequenos"] = int(len(pequenos))
R["pct_vendors_pequenos"] = float(len(pequenos) / len(vendor_idx) * 100)

objetivo_peq = [(i, v & pequenos) for i, v in enumerate(verdad_idx)]
objetivo_peq = [(i, s) for i, s in objetivo_peq if s]
objetivo_peq_map = dict(objetivo_peq)
print(f"{miles(len(pequenos))} restaurantes pequeños ({UMBRAL_PEQUENO} pedidos o menos) · "
      f"{miles(len(objetivo_peq))} usuarios de prueba visitaron alguno")

def recall_pequenos(scores_fn, k=TOP_K, lotes=1_000):
    filas = np.array([i for i, _ in objetivo_peq])
    aciertos, total = 0, 0
    for ini in range(0, len(filas), lotes):
        sel = filas[ini:ini + lotes]
        S_lote = scores_fn(eval_pos[sel])
        top = np.argpartition(-S_lote, k, axis=1)[:, :k]
        for f, i in enumerate(sel):
            rel = objetivo_peq_map[i]
            aciertos += len(set(top[f].tolist()) & rel)
            total += len(rel)
    return aciertos / total * 100 if total else 0.0

frio = {
    "Popularidad": recall_pequenos(lambda idx: np.tile(pop_scores, (len(idx), 1))),
    "Contenido": recall_pequenos(scores_contenido),
    "CF ítem-ítem": recall_pequenos(scores_item_item),
    "ALS": recall_pequenos(scores_als),
}
R["arranque_frio"] = {k: float(v) for k, v in frio.items()}
for k, v in frio.items():
    print(f"{k:16s} recall@10 sobre restaurantes pequeños: {pct(v)}")

fig, ax = plt.subplots(figsize=(6.2, 3.2))
nombres = list(frio); valores = [frio[n] for n in nombres]
mejor = int(np.argmax(valores))
ax.bar(range(len(nombres)), valores,
       color=[JITOMATE if i == mejor else COBALTO for i in range(len(nombres))], width=0.65)
for i, v in enumerate(valores):
    ax.text(i, v + max(valores) * 0.03, f"{v:.2f}", ha="center", color=CARBON,
            fontsize=9, fontweight="bold")
ax.set_xticks(range(len(nombres))); ax.set_xticklabels(nombres)
ax.set_ylabel("Recall@10 (%)")
R["mejor_arranque_frio"] = nombres[mejor]
titulo_ax(ax, f"{nombres[mejor]} descubre mejor a los restaurantes de {UMBRAL_PEQUENO} pedidos o menos")
ax.grid(axis="y"); ax.set_axisbelow(True)
guardar(fig, "05_arranque_frio"); plt.show()

1,950 restaurantes pequeños (50 pedidos o menos) · 1,429 usuarios de prueba visitaron alguno


Popularidad      recall@10 sobre restaurantes pequeños: 0.00%
Contenido        recall@10 sobre restaurantes pequeños: 7.12%
CF ítem-ítem     recall@10 sobre restaurantes pequeños: 4.21%
ALS              recall@10 sobre restaurantes pequeños: 0.00%
→ figs/05_arranque_frio.png


## 10. Mapa de gustos (t-SNE sobre los factores de ALS)

In [30]:
from sklearn.manifold import TSNE

n_mapa = 800 if SMOKE else 2_500
candidatos = np.argsort(-pop)[:n_mapa]
emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED,
           max_iter=500 if SMOKE else 1000).fit_transform(V[candidatos])

coc_top = pd.Series(cocina_v[candidatos]).value_counts().head(4).index.tolist()
fig, ax = plt.subplots(figsize=(6.6, 5.2))
ax.scatter(emb[:, 0], emb[:, 1], s=8, color=HAIRLINE, alpha=0.7, linewidths=0)
for color, c in zip(SERIE, coc_top):
    m = cocina_v[candidatos] == c
    ax.scatter(emb[m, 0], emb[m, 1], s=16, color=color, label=c, linewidths=0)
ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc="lower right", fontsize=9)
titulo_ax(ax, "Los factores latentes agrupan por tipo de cocina")
guardar(fig, "06_mapa_tsne"); plt.show()

→ figs/06_mapa_tsne.png


## 11. Resultados guardados

In [31]:
R["generado"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
with open(OUT / "resultados.json", "w", encoding="utf-8") as f:
    json.dump(R, f, ensure_ascii=False, indent=2)
tabla.to_csv(OUT / "tabla_modelos.csv", index=False)
print("Guardado en", (OUT / "resultados.json").resolve())
tabla.round(2)

Guardado en /home/claude/work/run3/export/resultados.json


,modelo,precision@10,recall@10,ndcg@10,recall repetidos,recall nuevos,cobertura,segundos
0,Híbrido repetir/explorar,5.91,36.26,28.76,96.65,6.36,86.14,15.27
1,Caminata aleatoria (PPR),5.90,36.25,28.75,96.49,6.39,86.06,12.07
2,Lo de siempre,5.17,31.11,26.29,95.89,0.00,85.70,1.47
3,ALS (factores latentes),3.64,24.01,16.12,48.26,9.52,36.53,1.29
4,Contenido,2.59,17.99,12.66,42.13,2.88,99.43,1.32
5,CF ítem-ítem,1.77,9.92,5.67,12.73,9.92,93.01,3.21
6,SVD truncado,1.06,6.11,3.59,12.18,3.67,8.68,1.22
7,Popularidad reciente (14 días),0.12,0.69,0.39,0.42,0.87,0.13,1.11
8,Popularidad,0.00,0.00,0.00,0.00,0.00,0.13,1.19
